In [1]:
# ============================================================
# DEEPSHIELD-AI — VIDEO MODEL IMPROVEMENT
# STEP 1: IMPORTS & CONFIGURATION
# ============================================================

import os
import cv2
import copy
import time
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from PIL import Image

from torch.utils.data import (
    Dataset,
    DataLoader
)

from torchvision import (
    models,
    transforms
)

from sklearn.model_selection import train_test_split

print("=" * 70)
print("DEEPSHIELD-AI — VIDEO MODEL IMPROVEMENT")
print("=" * 70)

# ============================================================
# DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

# ============================================================
# REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seed:", SEED)

# ============================================================
# VIDEO CONFIGURATION
# ============================================================

NUM_FRAMES = 16
FRAME_SIZE = 224
NUM_CLASSES = 2

CNN_FEATURES = 512
LSTM_HIDDEN_SIZE = 256

BATCH_SIZE = 2
NUM_EPOCHS = 15

print("\nVideo configuration:")
print("Frames/video :", NUM_FRAMES)
print("Frame size   :", FRAME_SIZE)
print("Batch size   :", BATCH_SIZE)
print("Epochs       :", NUM_EPOCHS)

DEEPSHIELD-AI — VIDEO MODEL IMPROVEMENT
Device: cuda
Seed: 42

Video configuration:
Frames/video : 16
Frame size   : 224
Batch size   : 2
Epochs       : 15


In [2]:
# ============================================================
# STEP 2 — LOAD SDFVD DATASET
# ============================================================

from datasets import load_dataset

sdfvd_dataset = load_dataset(
    "Hemgg/SDFVD-video-dataset",
    split="train"
)

print("=" * 70)
print("SDFVD DATASET")
print("=" * 70)

print("Total videos:", len(sdfvd_dataset))
print("Features:", sdfvd_dataset.features)

Repo card metadata block was not found. Setting CardData to empty.


Resolving data files:   0%|          | 0/106 [00:00<?, ?it/s]

SDFVD DATASET
Total videos: 106
Features: {'video': Video(decode=True, stream_index=None, dimension_order='NCHW', num_ffmpeg_threads=1, device='cpu', seek_mode='exact'), 'label': ClassLabel(names=['Fake', 'Real'])}


In [3]:
# ============================================================
# STEP 3 — EXTRACT VIDEO METADATA
# ============================================================

video_metadata = []

for i in range(len(sdfvd_dataset)):

    raw_video = (
        sdfvd_dataset.data
        .column("video")[i]
        .as_py()
    )

    video_path = raw_video["path"]
    label = int(sdfvd_dataset["label"][i])

    video_metadata.append({
        "index": i,
        "video_path": video_path,
        "label": label
    })

print("=" * 70)
print("VIDEO METADATA")
print("=" * 70)

print("Total:", len(video_metadata))

print("\nSample:")
print(video_metadata[0])

VIDEO METADATA
Total: 106

Sample:
{'index': 0, 'video_path': 'C:\\Users\\saksh\\.cache\\huggingface\\hub\\datasets--Hemgg--SDFVD-video-dataset\\snapshots\\11239a51248ad96a767460b7613cbf3b99b2f547\\Fake\\vs1.mp4', 'label': 0}


In [4]:
# ============================================================
# STEP 4 — LABEL VERIFICATION
# ============================================================

labels = np.array([
    item["label"]
    for item in video_metadata
])

print("=" * 70)
print("LABEL DISTRIBUTION")
print("=" * 70)

print("FAKE (0):", np.sum(labels == 0))
print("REAL (1):", np.sum(labels == 1))
print("TOTAL   :", len(labels))

LABEL DISTRIBUTION
FAKE (0): 53
REAL (1): 53
TOTAL   : 106


In [5]:
# ============================================================
# STEP 5 — RECREATE EXACT TRAIN / VAL / TEST SPLIT
# ============================================================

all_indices = np.arange(
    len(video_metadata)
)

train_indices, temp_indices = train_test_split(
    all_indices,
    test_size=0.30,
    random_state=SEED,
    stratify=labels
)

temp_labels = labels[temp_indices]

val_indices, test_indices = train_test_split(
    temp_indices,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_labels
)

print("=" * 70)
print("DATA SPLIT")
print("=" * 70)

print("Train      :", len(train_indices))
print("Validation :", len(val_indices))
print("Test       :", len(test_indices))

print(
    "Total      :",
    len(train_indices)
    + len(val_indices)
    + len(test_indices)
)

DATA SPLIT
Train      : 74
Validation : 16
Test       : 16
Total      : 106


In [6]:
# ============================================================
# STEP 6 — TRAINING TRANSFORM
# ============================================================

train_transform = transforms.Compose([
    transforms.Resize((FRAME_SIZE, FRAME_SIZE)),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomApply(
        [
            transforms.ColorJitter(
                brightness=0.15,
                contrast=0.15,
                saturation=0.10,
                hue=0.02
            )
        ],
        p=0.5
    ),

    transforms.RandomApply(
        [
            transforms.RandomRotation(
                degrees=5
            )
        ],
        p=0.3
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("=" * 70)
print("TRAINING TRANSFORM READY")
print("=" * 70)

print("Frame size:", FRAME_SIZE)
print("Augmentation: ENABLED")
print("Output tensor: [3, 224, 224]")

TRAINING TRANSFORM READY
Frame size: 224
Augmentation: ENABLED
Output tensor: [3, 224, 224]


In [7]:
# ============================================================
# STEP 7 — VALIDATION / TEST TRANSFORM
# ============================================================

eval_transform = transforms.Compose([
    transforms.Resize((FRAME_SIZE, FRAME_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("=" * 70)
print("EVALUATION TRANSFORM READY")
print("=" * 70)

print("Augmentation: DISABLED")
print("Output tensor: [3, 224, 224]")

EVALUATION TRANSFORM READY
Augmentation: DISABLED
Output tensor: [3, 224, 224]


In [8]:
# ============================================================
# STEP 8 — IMPROVED VIDEO DATASET
# ============================================================

class ImprovedVideoDataset(Dataset):

    def __init__(
        self,
        metadata,
        indices,
        num_frames=16,
        transform=None
    ):
        self.metadata = metadata
        self.indices = list(indices)
        self.num_frames = num_frames
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):

        metadata_index = self.indices[idx]

        item = self.metadata[metadata_index]

        video_path = item["video_path"]
        label = int(item["label"])

        cap = cv2.VideoCapture(video_path)

        if not cap.isOpened():
            raise RuntimeError(
                f"Could not open video:\n{video_path}"
            )

        total_frames = int(
            cap.get(cv2.CAP_PROP_FRAME_COUNT)
        )

        if total_frames <= 0:
            cap.release()

            raise RuntimeError(
                f"Invalid video:\n{video_path}"
            )

        frame_indices = np.linspace(
            0,
            total_frames - 1,
            self.num_frames,
            dtype=int
        )

        frames = []

        for frame_index in frame_indices:

            cap.set(
                cv2.CAP_PROP_POS_FRAMES,
                int(frame_index)
            )

            success, frame = cap.read()

            if not success:
                continue

            frame = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB
            )

            image = Image.fromarray(frame)

            if self.transform is not None:
                image = self.transform(image)

            frames.append(image)

        cap.release()

        if len(frames) == 0:
            raise RuntimeError(
                f"No frames extracted:\n{video_path}"
            )

        # Duplicate last valid frame if necessary
        while len(frames) < self.num_frames:
            frames.append(
                frames[-1].clone()
            )

        frames = frames[:self.num_frames]

        video_tensor = torch.stack(frames)

        return video_tensor, label


print("=" * 70)
print("IMPROVED VIDEO DATASET CREATED")
print("=" * 70)

IMPROVED VIDEO DATASET CREATED


In [9]:
# ============================================================
# STEP 9 — CREATE DATASETS
# ============================================================

improved_train_dataset = ImprovedVideoDataset(
    metadata=video_metadata,
    indices=train_indices,
    num_frames=NUM_FRAMES,
    transform=train_transform
)

improved_val_dataset = ImprovedVideoDataset(
    metadata=video_metadata,
    indices=val_indices,
    num_frames=NUM_FRAMES,
    transform=eval_transform
)

improved_test_dataset = ImprovedVideoDataset(
    metadata=video_metadata,
    indices=test_indices,
    num_frames=NUM_FRAMES,
    transform=eval_transform
)

print("=" * 70)
print("DATASETS CREATED")
print("=" * 70)

print(
    "Train      :",
    len(improved_train_dataset)
)

print(
    "Validation :",
    len(improved_val_dataset)
)

print(
    "Test       :",
    len(improved_test_dataset)
)

DATASETS CREATED
Train      : 74
Validation : 16
Test       : 16


In [10]:
# ============================================================
# STEP 10 — DATASET SAMPLE CHECK
# ============================================================

sample_video, sample_label = (
    improved_train_dataset[0]
)

print("=" * 70)
print("IMPROVED DATASET SAMPLE CHECK")
print("=" * 70)

print("Video shape :", sample_video.shape)
print("Label       :", sample_label)
print("Dtype       :", sample_video.dtype)
print("Min value   :", sample_video.min().item())
print("Max value   :", sample_video.max().item())

IMPROVED DATASET SAMPLE CHECK
Video shape : torch.Size([16, 3, 224, 224])
Label       : 1
Dtype       : torch.float32
Min value   : -2.1179039478302
Max value   : 2.640000104904175


In [11]:
# ============================================================
# STEP 11 — CREATE DATALOADERS
# ============================================================

train_loader = DataLoader(
    improved_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    improved_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    improved_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("=" * 70)
print("DATALOADERS READY")
print("=" * 70)

print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))
print("Test batches :", len(test_loader))

DATALOADERS READY
Train batches: 37
Val batches  : 8
Test batches : 8


In [12]:
# ============================================================
# STEP 12 — PRETRAINED RESNET18
# ============================================================

weights = models.ResNet18_Weights.DEFAULT

resnet = models.resnet18(
    weights=weights
)

# Remove original ImageNet classifier
resnet.fc = nn.Identity()

resnet = resnet.to(DEVICE)

print("=" * 70)
print("RESNET18 READY")
print("=" * 70)

print("Feature size:", CNN_FEATURES)
print("Device:", DEVICE)

RESNET18 READY
Feature size: 512
Device: cuda


In [13]:
# ============================================================
# STEP 13 — FREEZE RESNET
# ============================================================

for parameter in resnet.parameters():
    parameter.requires_grad = False

print("=" * 70)
print("RESNET18 FROZEN")
print("=" * 70)

total_params = sum(
    p.numel()
    for p in resnet.parameters()
)

trainable_params = sum(
    p.numel()
    for p in resnet.parameters()
    if p.requires_grad
)

print("Total parameters     :", total_params)
print("Trainable parameters :", trainable_params)

RESNET18 FROZEN
Total parameters     : 11176512
Trainable parameters : 0


In [14]:
# ============================================================
# STEP 14 — VIDEO RESNET18 + LSTM MODEL
# ============================================================

class VideoResNet18LSTM(nn.Module):

    def __init__(
        self,
        cnn,
        cnn_features=512,
        hidden_size=256,
        num_layers=1,
        num_classes=2,
        dropout=0.4
    ):
        super().__init__()

        self.cnn = cnn

        self.lstm = nn.LSTM(
            input_size=cnn_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.0
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.classifier = nn.Linear(
            hidden_size,
            num_classes
        )

    def forward(self, x):

        # x:
        # [B, T, C, H, W]

        batch_size, time_steps, C, H, W = x.shape

        # Combine batch and temporal dimensions
        x = x.view(
            batch_size * time_steps,
            C,
            H,
            W
        )

        # CNN feature extraction
        with torch.no_grad():
            features = self.cnn(x)

        # [B*T, 512]
        features = features.view(
            batch_size,
            time_steps,
            -1
        )

        # Temporal modeling
        lstm_out, _ = self.lstm(
            features
        )

        # Last temporal feature
        last_output = lstm_out[:, -1, :]

        last_output = self.dropout(
            last_output
        )

        output = self.classifier(
            last_output
        )

        return output

In [15]:
# ============================================================
# STEP 15 — INITIALIZE VIDEO MODEL
# ============================================================

video_model = VideoResNet18LSTM(
    cnn=resnet,
    cnn_features=CNN_FEATURES,
    hidden_size=LSTM_HIDDEN_SIZE,
    num_layers=1,
    num_classes=NUM_CLASSES,
    dropout=0.4
).to(DEVICE)

print("=" * 70)
print("VIDEO MODEL CREATED")
print("=" * 70)

total_params = sum(
    p.numel()
    for p in video_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in video_model.parameters()
    if p.requires_grad
)

print("Total parameters     :", total_params)
print("Trainable parameters :", trainable_params)

VIDEO MODEL CREATED
Total parameters     : 11965506
Trainable parameters : 788994


In [16]:
# ============================================================
# STEP 16 — FORWARD PASS CHECK
# ============================================================

sample_batch, sample_labels = next(
    iter(train_loader)
)

print("=" * 70)
print("FORWARD PASS CHECK")
print("=" * 70)

print("Input shape :", sample_batch.shape)
print("Labels shape:", sample_labels.shape)

sample_batch = sample_batch.to(DEVICE)

with torch.no_grad():
    sample_output = video_model(sample_batch)

print("Output shape:", sample_output.shape)
print("Output:", sample_output)

FORWARD PASS CHECK
Input shape : torch.Size([2, 16, 3, 224, 224])
Labels shape: torch.Size([2])
Output shape: torch.Size([2, 2])
Output: tensor([[-0.0490, -0.1128],
        [-0.0424, -0.0129]], device='cuda:0')


In [17]:
# ============================================================
# STEP 17 — LOSS FUNCTION CHECK
# ============================================================

criterion = nn.CrossEntropyLoss()

sample_labels = sample_labels.to(DEVICE)

sample_loss = criterion(
    sample_output,
    sample_labels
)

print("=" * 70)
print("LOSS FUNCTION CHECK")
print("=" * 70)

print("Output shape :", sample_output.shape)
print("Label shape  :", sample_labels.shape)
print("Loss         :", sample_loss.item())

LOSS FUNCTION CHECK
Output shape : torch.Size([2, 2])
Label shape  : torch.Size([2])
Loss         : 0.7020093202590942


In [18]:
# ============================================================
# STEP 18 — OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        video_model.parameters()
    ),
    lr=1e-4,
    weight_decay=1e-3
)

print("=" * 70)
print("OPTIMIZER READY")
print("=" * 70)

print("Optimizer : AdamW")
print("LR        : 0.0001")
print("Weight decay:", 0.001)

OPTIMIZER READY
Optimizer : AdamW
LR        : 0.0001
Weight decay: 0.001


In [19]:
# ============================================================
# STEP 19 — LEARNING RATE SCHEDULER
# ============================================================

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

print("=" * 70)
print("SCHEDULER READY")
print("=" * 70)

print("Scheduler: ReduceLROnPlateau")
print("Factor   : 0.5")
print("Patience : 2")

SCHEDULER READY
Scheduler: ReduceLROnPlateau
Factor   : 0.5
Patience : 2


In [20]:
# ============================================================
# STEP 20 — TRAINING CONFIGURATION
# ============================================================

BEST_MODEL_PATH = os.path.join(
    "models",
    "video",
    "video_resnet18_lstm_improved_best.pth"
)

os.makedirs(
    os.path.dirname(BEST_MODEL_PATH),
    exist_ok=True
)

best_val_accuracy = 0.0
best_val_loss = float("inf")

patience_counter = 0
EARLY_STOPPING_PATIENCE = 5

print("=" * 70)
print("TRAINING CONFIGURATION")
print("=" * 70)

print("Epochs              :", NUM_EPOCHS)
print("Best model path     :", BEST_MODEL_PATH)
print("Early stopping      :", EARLY_STOPPING_PATIENCE)

TRAINING CONFIGURATION
Epochs              : 15
Best model path     : models\video\video_resnet18_lstm_improved_best.pth
Early stopping      : 5


In [21]:
# ============================================================
# STEP 21 — TRAINING FUNCTION
# ============================================================

def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for videos, labels in loader:

        videos = videos.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad()

        outputs = model(videos)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        running_loss += (
            loss.item() * labels.size(0)
        )

        predictions = outputs.argmax(
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy


print("=" * 70)
print("TRAINING FUNCTION READY")
print("=" * 70)

TRAINING FUNCTION READY


In [22]:
# ============================================================
# STEP 22 — VALIDATION FUNCTION
# ============================================================

def validate_one_epoch(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for videos, labels in loader:

            videos = videos.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model(videos)

            loss = criterion(
                outputs,
                labels
            )

            running_loss += (
                loss.item() * labels.size(0)
            )

            predictions = outputs.argmax(
                dim=1
            )

            correct += (
                predictions == labels
            ).sum().item()

            total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy


print("=" * 70)
print("VALIDATION FUNCTION READY")
print("=" * 70)

VALIDATION FUNCTION READY


In [23]:
# ============================================================
# STEP 23 — TRAINING HISTORY
# ============================================================

history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
    "learning_rate": []
}

print("=" * 70)
print("TRAINING HISTORY INITIALIZED")
print("=" * 70)

TRAINING HISTORY INITIALIZED


In [24]:
# ============================================================
# STEP 24 — IMPROVED MODEL TRAINING
# ============================================================

print("=" * 70)
print("DEEPSHIELD-AI — IMPROVED VIDEO TRAINING")
print("=" * 70)

best_val_accuracy = 0.0
best_val_loss = float("inf")
patience_counter = 0

for epoch in range(NUM_EPOCHS):

    start_time = time.time()

    train_loss, train_accuracy = train_one_epoch(
        video_model,
        train_loader,
        criterion,
        optimizer,
        DEVICE
    )

    val_loss, val_accuracy = validate_one_epoch(
        video_model,
        val_loader,
        criterion,
        DEVICE
    )

    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(
        train_loss
    )

    history["train_accuracy"].append(
        train_accuracy
    )

    history["val_loss"].append(
        val_loss
    )

    history["val_accuracy"].append(
        val_accuracy
    )

    history["learning_rate"].append(
        current_lr
    )

    elapsed = (
        time.time() - start_time
    ) / 60

    print("\n" + "=" * 70)
    print(
        f"EPOCH {epoch + 1}/{NUM_EPOCHS}"
    )
    print("=" * 70)

    print(
        f"Training Loss     : {train_loss:.4f}"
    )

    print(
        f"Training Accuracy : {train_accuracy:.4f}"
    )

    print(
        f"Validation Loss   : {val_loss:.4f}"
    )

    print(
        f"Validation Acc.   : {val_accuracy:.4f}"
    )

    print(
        f"Learning Rate     : {current_lr:.7f}"
    )

    print(
        f"Time              : {elapsed:.2f} min"
    )

    # --------------------------------------------------------
    # SAVE BEST MODEL
    # --------------------------------------------------------

    if (
        val_accuracy > best_val_accuracy
        or (
            val_accuracy == best_val_accuracy
            and val_loss < best_val_loss
        )
    ):

        best_val_accuracy = val_accuracy
        best_val_loss = val_loss

        torch.save(
            {
                "model_state_dict":
                    video_model.state_dict(),

                "val_accuracy":
                    best_val_accuracy,

                "val_loss":
                    best_val_loss,

                "epoch":
                    epoch + 1
            },
            BEST_MODEL_PATH
        )

        patience_counter = 0

        print("★ BEST MODEL SAVED")

    else:

        patience_counter += 1

    # --------------------------------------------------------
    # EARLY STOPPING
    # --------------------------------------------------------

    if (
        patience_counter
        >= EARLY_STOPPING_PATIENCE
    ):

        print(
            "\nEarly stopping triggered."
        )

        break


print("\n" + "=" * 70)
print("IMPROVED TRAINING COMPLETE")
print("=" * 70)

print(
    f"Best Validation Accuracy : "
    f"{best_val_accuracy:.4f}"
)

print(
    f"Best Validation Loss     : "
    f"{best_val_loss:.4f}"
)

print(
    f"Best Model Saved At       : "
    f"{BEST_MODEL_PATH}"
)

DEEPSHIELD-AI — IMPROVED VIDEO TRAINING

EPOCH 1/15
Training Loss     : 0.6977
Training Accuracy : 0.4865
Validation Loss   : 0.7346
Validation Acc.   : 0.3125
Learning Rate     : 0.0001000
Time              : 0.63 min
★ BEST MODEL SAVED

EPOCH 2/15
Training Loss     : 0.6724
Training Accuracy : 0.5676
Validation Loss   : 0.7595
Validation Acc.   : 0.2500
Learning Rate     : 0.0001000
Time              : 0.58 min

EPOCH 3/15
Training Loss     : 0.6960
Training Accuracy : 0.5405
Validation Loss   : 0.7632
Validation Acc.   : 0.2500
Learning Rate     : 0.0001000
Time              : 0.58 min

EPOCH 4/15
Training Loss     : 0.6953
Training Accuracy : 0.5135
Validation Loss   : 0.7740
Validation Acc.   : 0.2500
Learning Rate     : 0.0000500
Time              : 0.58 min

EPOCH 5/15
Training Loss     : 0.6833
Training Accuracy : 0.5676
Validation Loss   : 0.7765
Validation Acc.   : 0.1875
Learning Rate     : 0.0000500
Time              : 0.56 min

EPOCH 6/15
Training Loss     : 0.6613
Trainin

In [25]:
# ============================================================
# STEP 25 — LOAD BEST MODEL
# ============================================================

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE
)

video_model.load_state_dict(
    checkpoint["model_state_dict"]
)

video_model = video_model.to(DEVICE)

print("=" * 70)
print("BEST MODEL LOADED")
print("=" * 70)

print(
    "Best epoch:",
    checkpoint["epoch"]
)

print(
    "Best validation accuracy:",
    checkpoint["val_accuracy"]
)

print(
    "Best validation loss:",
    checkpoint["val_loss"]
)


BEST MODEL LOADED
Best epoch: 6
Best validation accuracy: 0.375
Best validation loss: 0.7836741581559181


In [26]:
# ============================================================
# STEP 26 — BEST MODEL VALIDATION CHECK
# ============================================================

best_val_loss, best_val_accuracy = (
    validate_one_epoch(
        video_model,
        val_loader,
        criterion,
        DEVICE
    )
)

print("=" * 70)
print("BEST MODEL — VALIDATION RESULT")
print("=" * 70)

print(
    f"Validation Loss     : "
    f"{best_val_loss:.4f}"
)

print(
    f"Validation Accuracy : "
    f"{best_val_accuracy:.4f}"
)

BEST MODEL — VALIDATION RESULT
Validation Loss     : 0.7837
Validation Accuracy : 0.3750


In [27]:
# ============================================================
# STEP 27 — TEST EVALUATION FUNCTION
# ============================================================

def evaluate_test_set(
    model,
    loader,
    device
):

    model.eval()

    all_labels = []
    all_predictions = []
    all_probabilities = []

    with torch.no_grad():

        for videos, labels in loader:

            videos = videos.to(
                device,
                non_blocking=True
            )

            outputs = model(videos)

            probabilities = torch.softmax(
                outputs,
                dim=1
            )

            predictions = outputs.argmax(
                dim=1
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_probabilities.extend(
                probabilities.cpu().numpy()
            )

    return (
        np.array(all_labels),
        np.array(all_predictions),
        np.array(all_probabilities)
    )


print("=" * 70)
print("TEST EVALUATION FUNCTION READY")
print("=" * 70)

TEST EVALUATION FUNCTION READY


In [28]:
# ============================================================
# STEP 28 — IMPROVED MODEL TEST PREDICTIONS
# ============================================================

improved_labels, improved_predictions, improved_probabilities = (
    evaluate_test_set(
        video_model,
        test_loader,
        DEVICE
    )
)

print("=" * 70)
print("IMPROVED MODEL — TEST PREDICTIONS")
print("=" * 70)

print(
    "Number of test videos:",
    len(improved_labels)
)

print(
    "True labels:",
    improved_labels
)

print(
    "Predictions:",
    improved_predictions
)

IMPROVED MODEL — TEST PREDICTIONS
Number of test videos: 16
True labels: [0 0 0 1 1 0 0 1 1 1 1 1 0 0 0 1]
Predictions: [0 1 1 0 0 0 1 0 0 0 1 0 0 0 1 0]


In [29]:
# ============================================================
# STEP 29 — IMPROVED TEST METRICS
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

improved_accuracy = accuracy_score(
    improved_labels,
    improved_predictions
)

improved_precision = precision_score(
    improved_labels,
    improved_predictions,
    zero_division=0
)

improved_recall = recall_score(
    improved_labels,
    improved_predictions,
    zero_division=0
)

improved_f1 = f1_score(
    improved_labels,
    improved_predictions,
    zero_division=0
)

print("=" * 70)
print("DEEPSHIELD-AI — IMPROVED VIDEO TEST RESULTS")
print("=" * 70)

print(
    f"Accuracy  : {improved_accuracy:.4f}"
)

print(
    f"Precision : {improved_precision:.4f}"
)

print(
    f"Recall    : {improved_recall:.4f}"
)

print(
    f"F1 Score  : {improved_f1:.4f}"
)

DEEPSHIELD-AI — IMPROVED VIDEO TEST RESULTS
Accuracy  : 0.3125
Precision : 0.2000
Recall    : 0.1250
F1 Score  : 0.1538


In [30]:
# ============================================================
# STEP 30 — IMPROVED CLASSIFICATION REPORT
# ============================================================

improved_cm = confusion_matrix(
    improved_labels,
    improved_predictions,
    labels=[0, 1]
)

print("=" * 70)
print("IMPROVED VIDEO CONFUSION MATRIX")
print("=" * 70)

print("              Predicted")
print("             FAKE  REAL")

print(
    f"Actual FAKE   "
    f"{improved_cm[0,0]:>3}   "
    f"{improved_cm[0,1]:>3}"
)

print(
    f"Actual REAL   "
    f"{improved_cm[1,0]:>3}   "
    f"{improved_cm[1,1]:>3}"
)

print("\n" + "=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        improved_labels,
        improved_predictions,
        labels=[0, 1],
        target_names=["FAKE", "REAL"],
        zero_division=0
    )
)

IMPROVED VIDEO CONFUSION MATRIX
              Predicted
             FAKE  REAL
Actual FAKE     4     4
Actual REAL     7     1

CLASSIFICATION REPORT
              precision    recall  f1-score   support

        FAKE       0.36      0.50      0.42         8
        REAL       0.20      0.12      0.15         8

    accuracy                           0.31        16
   macro avg       0.28      0.31      0.29        16
weighted avg       0.28      0.31      0.29        16



In [31]:
# ============================================================
# STEP 31 — CLASS WEIGHTS
# ============================================================

train_labels = np.array([
    video_metadata[i]["label"]
    for i in train_indices
])

class_counts = np.bincount(
    train_labels,
    minlength=NUM_CLASSES
)

class_weights = (
    len(train_labels)
    / (
        NUM_CLASSES * class_counts
    )
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=DEVICE
)

print("=" * 70)
print("CLASS BALANCE")
print("=" * 70)

print("FAKE count :", class_counts[0])
print("REAL count :", class_counts[1])

print("\nClass weights:")
print("FAKE:", class_weights[0].item())
print("REAL:", class_weights[1].item())

CLASS BALANCE
FAKE count : 37
REAL count : 37

Class weights:
FAKE: 1.0
REAL: 1.0


In [32]:
# ============================================================
# STEP 32 — BALANCED CROSS ENTROPY LOSS
# ============================================================

criterion_balanced = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.05
)

print("=" * 70)
print("BALANCED LOSS READY")
print("=" * 70)

print("Loss: Weighted CrossEntropyLoss")
print("Label smoothing: 0.05")

BALANCED LOSS READY
Loss: Weighted CrossEntropyLoss
Label smoothing: 0.05


In [33]:
# ============================================================
# STEP 33 — FRESH VIDEO MODEL
# ============================================================

resnet_balanced = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

resnet_balanced.fc = nn.Identity()

for parameter in resnet_balanced.parameters():
    parameter.requires_grad = False

resnet_balanced = resnet_balanced.to(DEVICE)

balanced_model = VideoResNet18LSTM(
    cnn=resnet_balanced,
    cnn_features=CNN_FEATURES,
    hidden_size=LSTM_HIDDEN_SIZE,
    num_layers=1,
    num_classes=NUM_CLASSES,
    dropout=0.5
).to(DEVICE)

print("=" * 70)
print("FRESH BALANCED MODEL CREATED")
print("=" * 70)

print(
    "Total parameters     :",
    sum(
        p.numel()
        for p in balanced_model.parameters()
    )
)

print(
    "Trainable parameters :",
    sum(
        p.numel()
        for p in balanced_model.parameters()
        if p.requires_grad
    )
)

FRESH BALANCED MODEL CREATED
Total parameters     : 11965506
Trainable parameters : 788994


In [34]:
# ============================================================
# STEP 34 — BALANCED MODEL OPTIMIZER
# ============================================================

balanced_optimizer = torch.optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        balanced_model.parameters()
    ),
    lr=5e-5,
    weight_decay=1e-3
)

balanced_scheduler = (
    torch.optim.lr_scheduler.ReduceLROnPlateau(
        balanced_optimizer,
        mode="min",
        factor=0.5,
        patience=2
    )
)

print("=" * 70)
print("BALANCED MODEL OPTIMIZER READY")
print("=" * 70)

print("Learning rate : 0.00005")
print("Weight decay  : 0.001")

BALANCED MODEL OPTIMIZER READY
Learning rate : 0.00005
Weight decay  : 0.001


In [35]:
# ============================================================
# STEP 35 — BALANCED MODEL FORWARD CHECK
# ============================================================

test_batch, test_labels = next(
    iter(train_loader)
)

test_batch = test_batch.to(DEVICE)
test_labels = test_labels.to(DEVICE)

balanced_model.eval()

with torch.no_grad():

    test_output = balanced_model(
        test_batch
    )

    test_loss = criterion_balanced(
        test_output,
        test_labels
    )

print("=" * 70)
print("BALANCED MODEL CHECK")
print("=" * 70)

print("Input shape :", test_batch.shape)
print("Output shape:", test_output.shape)
print("Loss        :", test_loss.item())

BALANCED MODEL CHECK
Input shape : torch.Size([2, 16, 3, 224, 224])
Output shape: torch.Size([2, 2])
Loss        : 0.6464366316795349


In [37]:
# ============================================================
# STEP 36 — BALANCED MODEL TRAINING FUNCTION
# ============================================================

def train_balanced_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device
):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for videos, labels in loader:

        videos = videos.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(videos)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        running_loss += (
            loss.item() * labels.size(0)
        )

        predictions = outputs.argmax(dim=1)

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    return (
        running_loss / total,
        correct / total
    )


print("=" * 70)
print("BALANCED TRAINING FUNCTION READY")
print("=" * 70)

BALANCED TRAINING FUNCTION READY


In [38]:
# ============================================================
# STEP 37 — BALANCED VALIDATION FUNCTION
# ============================================================

def validate_balanced(
    model,
    loader,
    criterion,
    device
):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for videos, labels in loader:

            videos = videos.to(device)
            labels = labels.to(device)

            outputs = model(videos)

            loss = criterion(
                outputs,
                labels
            )

            running_loss += (
                loss.item() * labels.size(0)
            )

            predictions = outputs.argmax(dim=1)

            correct += (
                predictions == labels
            ).sum().item()

            total += labels.size(0)

    return (
        running_loss / total,
        correct / total
    )


print("=" * 70)
print("BALANCED VALIDATION FUNCTION READY")
print("=" * 70)

BALANCED VALIDATION FUNCTION READY


In [39]:
# ============================================================
# STEP 38 — BALANCED TRAINING HISTORY
# ============================================================

balanced_history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
    "learning_rate": []
}

BALANCED_EPOCHS = 15

BALANCED_MODEL_PATH = os.path.join(
    "models",
    "video",
    "video_resnet18_lstm_balanced_best.pth"
)

os.makedirs(
    os.path.dirname(BALANCED_MODEL_PATH),
    exist_ok=True
)

best_balanced_accuracy = 0.0
best_balanced_loss = float("inf")
balanced_patience = 0

print("=" * 70)
print("BALANCED TRAINING CONFIGURED")
print("=" * 70)

print("Epochs:", BALANCED_EPOCHS)
print("Model path:", BALANCED_MODEL_PATH)

BALANCED TRAINING CONFIGURED
Epochs: 15
Model path: models\video\video_resnet18_lstm_balanced_best.pth


In [40]:
# ============================================================
# STEP 39 — BALANCED MODEL TRAINING
# ============================================================

print("=" * 70)
print("DEEPSHIELD-AI — BALANCED VIDEO TRAINING")
print("=" * 70)

for epoch in range(BALANCED_EPOCHS):

    start_time = time.time()

    train_loss, train_acc = train_balanced_epoch(
        balanced_model,
        train_loader,
        criterion_balanced,
        balanced_optimizer,
        DEVICE
    )

    val_loss, val_acc = validate_balanced(
        balanced_model,
        val_loader,
        criterion_balanced,
        DEVICE
    )

    balanced_scheduler.step(val_loss)

    current_lr = (
        balanced_optimizer
        .param_groups[0]["lr"]
    )

    balanced_history["train_loss"].append(
        train_loss
    )

    balanced_history["train_accuracy"].append(
        train_acc
    )

    balanced_history["val_loss"].append(
        val_loss
    )

    balanced_history["val_accuracy"].append(
        val_acc
    )

    balanced_history["learning_rate"].append(
        current_lr
    )

    elapsed = (
        time.time() - start_time
    ) / 60

    print("\n" + "=" * 70)
    print(f"EPOCH {epoch + 1}/{BALANCED_EPOCHS}")
    print("=" * 70)

    print(f"Training Loss     : {train_loss:.4f}")
    print(f"Training Accuracy : {train_acc:.4f}")
    print(f"Validation Loss   : {val_loss:.4f}")
    print(f"Validation Acc.   : {val_acc:.4f}")
    print(f"Learning Rate     : {current_lr:.7f}")
    print(f"Time              : {elapsed:.2f} min")

    if (
        val_acc > best_balanced_accuracy
        or (
            val_acc == best_balanced_accuracy
            and val_loss < best_balanced_loss
        )
    ):

        best_balanced_accuracy = val_acc
        best_balanced_loss = val_loss

        torch.save(
            {
                "model_state_dict":
                    balanced_model.state_dict(),

                "val_accuracy":
                    best_balanced_accuracy,

                "val_loss":
                    best_balanced_loss,

                "epoch":
                    epoch + 1
            },
            BALANCED_MODEL_PATH
        )

        balanced_patience = 0

        print("★ BEST BALANCED MODEL SAVED")

    else:
        balanced_patience += 1

    if balanced_patience >= 5:

        print("\nEarly stopping triggered.")

        break

print("\n" + "=" * 70)
print("BALANCED TRAINING COMPLETE")
print("=" * 70)

print(
    f"Best Validation Accuracy : "
    f"{best_balanced_accuracy:.4f}"
)

print(
    f"Best Validation Loss     : "
    f"{best_balanced_loss:.4f}"
)

DEEPSHIELD-AI — BALANCED VIDEO TRAINING

EPOCH 1/15
Training Loss     : 0.7133
Training Accuracy : 0.4189
Validation Loss   : 0.7013
Validation Acc.   : 0.5000
Learning Rate     : 0.0000500
Time              : 0.47 min
★ BEST BALANCED MODEL SAVED

EPOCH 2/15
Training Loss     : 0.7152
Training Accuracy : 0.4595
Validation Loss   : 0.7041
Validation Acc.   : 0.5000
Learning Rate     : 0.0000500
Time              : 0.46 min

EPOCH 3/15
Training Loss     : 0.6856
Training Accuracy : 0.5811
Validation Loss   : 0.7137
Validation Acc.   : 0.4375
Learning Rate     : 0.0000500
Time              : 0.46 min

EPOCH 4/15
Training Loss     : 0.7057
Training Accuracy : 0.5135
Validation Loss   : 0.7151
Validation Acc.   : 0.3750
Learning Rate     : 0.0000250
Time              : 0.47 min

EPOCH 5/15
Training Loss     : 0.6879
Training Accuracy : 0.5135
Validation Loss   : 0.7175
Validation Acc.   : 0.4375
Learning Rate     : 0.0000250
Time              : 0.46 min

EPOCH 6/15
Training Loss     : 0.691

In [41]:
# ============================================================
# STEP 41 — BALANCED MODEL TEST PREDICTIONS
# ============================================================

balanced_labels, balanced_predictions, balanced_probabilities = (
    evaluate_test_set(
        balanced_model,
        test_loader,
        DEVICE
    )
)

print("=" * 70)
print("BALANCED MODEL — TEST PREDICTIONS")
print("=" * 70)

print(
    "Number of test videos:",
    len(balanced_labels)
)

print(
    "True labels:",
    balanced_labels
)

print(
    "Predictions:",
    balanced_predictions
)

BALANCED MODEL — TEST PREDICTIONS
Number of test videos: 16
True labels: [0 0 0 1 1 0 0 1 1 1 1 1 0 0 0 1]
Predictions: [1 0 1 1 0 1 0 0 1 0 0 1 0 0 1 0]


In [42]:
# ============================================================
# STEP 42 — BALANCED TEST METRICS
# ============================================================

balanced_accuracy = accuracy_score(
    balanced_labels,
    balanced_predictions
)

balanced_precision = precision_score(
    balanced_labels,
    balanced_predictions,
    zero_division=0
)

balanced_recall = recall_score(
    balanced_labels,
    balanced_predictions,
    zero_division=0
)

balanced_f1 = f1_score(
    balanced_labels,
    balanced_predictions,
    zero_division=0
)

print("=" * 70)
print("DEEPSHIELD-AI — BALANCED VIDEO TEST RESULTS")
print("=" * 70)

print(f"Accuracy  : {balanced_accuracy:.4f}")
print(f"Precision : {balanced_precision:.4f}")
print(f"Recall    : {balanced_recall:.4f}")
print(f"F1 Score  : {balanced_f1:.4f}")

DEEPSHIELD-AI — BALANCED VIDEO TEST RESULTS
Accuracy  : 0.4375
Precision : 0.4286
Recall    : 0.3750
F1 Score  : 0.4000


In [43]:
# ============================================================
# STEP 43 — BALANCED CONFUSION MATRIX
# ============================================================

balanced_cm = confusion_matrix(
    balanced_labels,
    balanced_predictions,
    labels=[0, 1]
)

print("=" * 70)
print("BALANCED VIDEO CONFUSION MATRIX")
print("=" * 70)

print("              Predicted")
print("             FAKE  REAL")

print(
    f"Actual FAKE   "
    f"{balanced_cm[0,0]:>3}   "
    f"{balanced_cm[0,1]:>3}"
)

print(
    f"Actual REAL   "
    f"{balanced_cm[1,0]:>3}   "
    f"{balanced_cm[1,1]:>3}"
)

BALANCED VIDEO CONFUSION MATRIX
              Predicted
             FAKE  REAL
Actual FAKE     4     4
Actual REAL     5     3


In [44]:
# ============================================================
# STEP 44 — BALANCED CLASSIFICATION REPORT
# ============================================================

print("=" * 70)
print("BALANCED CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        balanced_labels,
        balanced_predictions,
        labels=[0, 1],
        target_names=["FAKE", "REAL"],
        zero_division=0
    )
)

BALANCED CLASSIFICATION REPORT
              precision    recall  f1-score   support

        FAKE       0.44      0.50      0.47         8
        REAL       0.43      0.38      0.40         8

    accuracy                           0.44        16
   macro avg       0.44      0.44      0.44        16
weighted avg       0.44      0.44      0.44        16



In [45]:
# ============================================================
# STEP 45 — EXPERIMENT COMPARISON
# ============================================================

print("=" * 70)
print("DEEPSHIELD-AI — VIDEO EXPERIMENT COMPARISON")
print("=" * 70)

print(
    "\nBaseline:"
)
print(
    "Accuracy : 0.3125"
)
print(
    "Macro F1 : 0.24"
)

print(
    "\nImproved/Augmentation:"
)
print(
    f"Accuracy : {improved_accuracy:.4f}"
)
print(
    f"F1       : {improved_f1:.4f}"
)

print(
    "\nBalanced:"
)
print(
    f"Accuracy : {balanced_accuracy:.4f}"
)
print(
    f"F1       : {balanced_f1:.4f}"
)

DEEPSHIELD-AI — VIDEO EXPERIMENT COMPARISON

Baseline:
Accuracy : 0.3125
Macro F1 : 0.24

Improved/Augmentation:
Accuracy : 0.3125
F1       : 0.1538

Balanced:
Accuracy : 0.4375
F1       : 0.4000


In [46]:
# ============================================================
# STEP 46 — BALANCED MODEL PREDICTION DISTRIBUTION
# ============================================================

print("=" * 70)
print("BALANCED MODEL — PREDICTION DISTRIBUTION")
print("=" * 70)

print(
    "True label counts:"
)

print(
    "FAKE (0):",
    np.sum(balanced_labels == 0)
)

print(
    "REAL (1):",
    np.sum(balanced_labels == 1)
)

print(
    "\nPredicted label counts:"
)

print(
    "FAKE (0):",
    np.sum(balanced_predictions == 0)
)

print(
    "REAL (1):",
    np.sum(balanced_predictions == 1)
)

BALANCED MODEL — PREDICTION DISTRIBUTION
True label counts:
FAKE (0): 8
REAL (1): 8

Predicted label counts:
FAKE (0): 9
REAL (1): 7


In [47]:
# ============================================================
# STEP 47 — BALANCED MODEL CONFIDENCE
# ============================================================

fake_confidence = balanced_probabilities[:, 0]
real_confidence = balanced_probabilities[:, 1]

print("=" * 70)
print("BALANCED MODEL — CONFIDENCE ANALYSIS")
print("=" * 70)

print(
    f"Average FAKE confidence : "
    f"{fake_confidence.mean():.4f}"
)

print(
    f"Average REAL confidence : "
    f"{real_confidence.mean():.4f}"
)

print(
    f"Minimum FAKE confidence : "
    f"{fake_confidence.min():.4f}"
)

print(
    f"Maximum FAKE confidence : "
    f"{fake_confidence.max():.4f}"
)

BALANCED MODEL — CONFIDENCE ANALYSIS
Average FAKE confidence : 0.5136
Average REAL confidence : 0.4864
Minimum FAKE confidence : 0.4148
Maximum FAKE confidence : 0.5752


In [48]:
# ============================================================
# STEP 48 — PER-VIDEO BALANCED PREDICTIONS
# ============================================================

print("=" * 70)
print("BALANCED VIDEO PREDICTIONS")
print("=" * 70)

for i in range(len(balanced_labels)):

    true_label = (
        "FAKE"
        if balanced_labels[i] == 0
        else "REAL"
    )

    predicted_label = (
        "FAKE"
        if balanced_predictions[i] == 0
        else "REAL"
    )

    fake_prob = balanced_probabilities[i][0]
    real_prob = balanced_probabilities[i][1]

    correct = (
        true_label == predicted_label
    )

    print(
        f"Video {i:02d} | "
        f"True: {true_label:<5} | "
        f"Pred: {predicted_label:<5} | "
        f"FAKE: {fake_prob:.4f} | "
        f"REAL: {real_prob:.4f} | "
        f"Correct: {correct}"
    )

BALANCED VIDEO PREDICTIONS
Video 00 | True: FAKE  | Pred: REAL  | FAKE: 0.4148 | REAL: 0.5852 | Correct: False
Video 01 | True: FAKE  | Pred: FAKE  | FAKE: 0.5223 | REAL: 0.4777 | Correct: True
Video 02 | True: FAKE  | Pred: REAL  | FAKE: 0.4467 | REAL: 0.5533 | Correct: False
Video 03 | True: REAL  | Pred: REAL  | FAKE: 0.4670 | REAL: 0.5330 | Correct: True
Video 04 | True: REAL  | Pred: FAKE  | FAKE: 0.5643 | REAL: 0.4357 | Correct: False
Video 05 | True: FAKE  | Pred: REAL  | FAKE: 0.4883 | REAL: 0.5117 | Correct: False
Video 06 | True: FAKE  | Pred: FAKE  | FAKE: 0.5118 | REAL: 0.4882 | Correct: True
Video 07 | True: REAL  | Pred: FAKE  | FAKE: 0.5752 | REAL: 0.4248 | Correct: False
Video 08 | True: REAL  | Pred: REAL  | FAKE: 0.4828 | REAL: 0.5172 | Correct: True
Video 09 | True: REAL  | Pred: FAKE  | FAKE: 0.5670 | REAL: 0.4330 | Correct: False
Video 10 | True: REAL  | Pred: FAKE  | FAKE: 0.5626 | REAL: 0.4374 | Correct: False
Video 11 | True: REAL  | Pred: REAL  | FAKE: 0.4519 |

In [49]:
# ============================================================
# STEP 49 — VIDEO RISK SCORE
# ============================================================

balanced_risk_scores = (
    balanced_probabilities[:, 0] * 100
)

print("=" * 70)
print("DEEPSHIELD-AI — BALANCED VIDEO RISK SCORES")
print("=" * 70)

for i, risk in enumerate(
    balanced_risk_scores
):

    if risk >= 70:
        level = "HIGH RISK"

    elif risk >= 40:
        level = "MEDIUM RISK"

    else:
        level = "LOW RISK"

    print(
        f"Video {i:02d} | "
        f"Risk Score: {risk:6.2f} | "
        f"{level}"
    )

DEEPSHIELD-AI — BALANCED VIDEO RISK SCORES
Video 00 | Risk Score:  41.48 | MEDIUM RISK
Video 01 | Risk Score:  52.23 | MEDIUM RISK
Video 02 | Risk Score:  44.67 | MEDIUM RISK
Video 03 | Risk Score:  46.70 | MEDIUM RISK
Video 04 | Risk Score:  56.43 | MEDIUM RISK
Video 05 | Risk Score:  48.83 | MEDIUM RISK
Video 06 | Risk Score:  51.18 | MEDIUM RISK
Video 07 | Risk Score:  57.52 | MEDIUM RISK
Video 08 | Risk Score:  48.28 | MEDIUM RISK
Video 09 | Risk Score:  56.70 | MEDIUM RISK
Video 10 | Risk Score:  56.26 | MEDIUM RISK
Video 11 | Risk Score:  45.19 | MEDIUM RISK
Video 12 | Risk Score:  57.25 | MEDIUM RISK
Video 13 | Risk Score:  56.56 | MEDIUM RISK
Video 14 | Risk Score:  47.32 | MEDIUM RISK
Video 15 | Risk Score:  55.21 | MEDIUM RISK


In [50]:
# ============================================================
# STEP 50 — RISK SUMMARY
# ============================================================

risk_levels = []

for risk in balanced_risk_scores:

    if risk >= 70:
        risk_levels.append(
            "HIGH RISK"
        )

    elif risk >= 40:
        risk_levels.append(
            "MEDIUM RISK"
        )

    else:
        risk_levels.append(
            "LOW RISK"
        )

risk_levels = np.array(
    risk_levels
)

print("=" * 70)
print("DEEPSHIELD-AI — VIDEO RISK SUMMARY")
print("=" * 70)

print(
    f"Average Risk Score : "
    f"{balanced_risk_scores.mean():.2f}"
)

print(
    f"Average Fake Confidence : "
    f"{fake_confidence.mean():.4f}"
)

print(
    f"Average Real Confidence : "
    f"{real_confidence.mean():.4f}"
)

print("\nRisk Levels:")

for level in [
    "HIGH RISK",
    "MEDIUM RISK",
    "LOW RISK"
]:

    print(
        f"{level:<12}: "
        f"{np.sum(risk_levels == level)}"
    )

DEEPSHIELD-AI — VIDEO RISK SUMMARY
Average Risk Score : 51.36
Average Fake Confidence : 0.5136
Average Real Confidence : 0.4864

Risk Levels:
HIGH RISK   : 0
MEDIUM RISK : 16
LOW RISK    : 0


In [51]:
# ============================================================
# STEP 51 — SAVE BALANCED EXPERIMENT RESULTS
# ============================================================

import os
import json
import numpy as np

# ------------------------------------------------------------
# LOAD BEST BALANCED CHECKPOINT DIRECTLY
# ------------------------------------------------------------

BALANCED_MODEL_PATH = os.path.join(
    "models",
    "video",
    "video_resnet18_lstm_balanced_best.pth"
)

if not os.path.exists(BALANCED_MODEL_PATH):
    raise FileNotFoundError(
        f"Balanced model not found:\n{BALANCED_MODEL_PATH}"
    )

balanced_checkpoint = torch.load(
    BALANCED_MODEL_PATH,
    map_location=DEVICE
)

# ------------------------------------------------------------
# GET BALANCED VALIDATION RESULTS
# ------------------------------------------------------------

balanced_validation_accuracy = float(
    balanced_checkpoint["val_accuracy"]
)

balanced_validation_loss = float(
    balanced_checkpoint["val_loss"]
)

# ------------------------------------------------------------
# GET BALANCED TEST RESULTS
# These variables were created during Steps 41–45
# ------------------------------------------------------------

balanced_test_accuracy = float(
    balanced_accuracy
)

balanced_test_precision = float(
    balanced_precision
)

balanced_test_recall = float(
    balanced_recall
)

balanced_test_f1 = float(
    balanced_f1
)

# ------------------------------------------------------------
# CONFIDENCE / RISK RESULTS
# ------------------------------------------------------------

average_fake_confidence = float(
    np.mean(balanced_probabilities[:, 0])
)

average_real_confidence = float(
    np.mean(balanced_probabilities[:, 1])
)

average_risk_score = float(
    np.mean(balanced_risk_scores)
)

# ------------------------------------------------------------
# FINAL RESULTS DICTIONARY
# ------------------------------------------------------------

video_experiment_results = {

    "baseline_test_accuracy": 0.3125,

    "balanced_validation_accuracy":
        balanced_validation_accuracy,

    "balanced_validation_loss":
        balanced_validation_loss,

    "balanced_test_accuracy":
        balanced_test_accuracy,

    "balanced_test_precision":
        balanced_test_precision,

    "balanced_test_recall":
        balanced_test_recall,

    "balanced_test_f1":
        balanced_test_f1,

    "average_fake_confidence":
        average_fake_confidence,

    "average_real_confidence":
        average_real_confidence,

    "average_risk_score":
        average_risk_score
}

# ------------------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------------------

print("=" * 70)
print("DEEPSHIELD-AI — VIDEO EXPERIMENT RESULTS")
print("=" * 70)

for key, value in video_experiment_results.items():

    print(
        f"{key:<35}: {value:.4f}"
        if isinstance(value, float)
        else f"{key:<35}: {value}"
    )

print("=" * 70)
print("STEP 51 COMPLETED SUCCESSFULLY")
print("=" * 70)

DEEPSHIELD-AI — VIDEO EXPERIMENT RESULTS
baseline_test_accuracy             : 0.3125
balanced_validation_accuracy       : 0.5000
balanced_validation_loss           : 0.7013
balanced_test_accuracy             : 0.4375
balanced_test_precision            : 0.4286
balanced_test_recall               : 0.3750
balanced_test_f1                   : 0.4000
average_fake_confidence            : 0.5136
average_real_confidence            : 0.4864
average_risk_score                 : 51.3624
STEP 51 COMPLETED SUCCESSFULLY


In [52]:
# ============================================================
# STEP 52 — SAVE VIDEO EXPERIMENT RESULTS
# ============================================================

RESULTS_PATH = os.path.join(
    "models",
    "video",
    "video_experiment_results.json"
)

with open(
    RESULTS_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        video_experiment_results,
        f,
        indent=4
    )

print("=" * 70)
print("VIDEO EXPERIMENT RESULTS SAVED")
print("=" * 70)
print("Path:", RESULTS_PATH)
print("Exists:", os.path.exists(RESULTS_PATH))

VIDEO EXPERIMENT RESULTS SAVED
Path: models\video\video_experiment_results.json
Exists: True


In [54]:
# ============================================================
# STEP 53 — VERIFY BALANCED MODEL CHECKPOINT
# ============================================================

print("=" * 70)
print("BALANCED MODEL CHECKPOINT VERIFICATION")
print("=" * 70)

print("Path:")
print(BALANCED_MODEL_PATH)

print("\nExists:")
print(os.path.exists(BALANCED_MODEL_PATH))

if os.path.exists(BALANCED_MODEL_PATH):

    size_mb = (
        os.path.getsize(BALANCED_MODEL_PATH)
        / (1024 ** 2)
    )

    print(
        f"\nCheckpoint size: {size_mb:.2f} MB"
    )

    checkpoint = torch.load(
        BALANCED_MODEL_PATH,
        map_location=DEVICE
    )

    print(
        "\nCheckpoint type:",
        type(checkpoint)
    )

    if isinstance(checkpoint, dict):

        print(
            "\nCheckpoint keys:"
        )

        for key in checkpoint.keys():
            print(" -", key)

BALANCED MODEL CHECKPOINT VERIFICATION
Path:
models\video\video_resnet18_lstm_balanced_best.pth

Exists:
True

Checkpoint size: 45.73 MB

Checkpoint type: <class 'dict'>

Checkpoint keys:
 - model_state_dict
 - val_accuracy
 - val_loss
 - epoch


In [55]:
# ============================================================
# STEP 54 — FINAL VIDEO MODEL METADATA
# ============================================================

video_model_metadata = {

    "project":
        "DeepShield-AI",

    "modality":
        "Video",

    "architecture":
        "ResNet18 + LSTM",

    "backbone":
        "ResNet18",

    "temporal_model":
        "LSTM",

    "num_frames":
        16,

    "frame_size":
        224,

    "input_channels":
        3,

    "num_classes":
        2,

    "classes": {
        "0": "FAKE",
        "1": "REAL"
    },

    "device":
        str(DEVICE),

    "checkpoint":
        BALANCED_MODEL_PATH,

    "validation_accuracy":
        float(balanced_validation_accuracy),

    "validation_loss":
        float(balanced_validation_loss),

    "test_accuracy":
        float(balanced_test_accuracy),

    "test_precision":
        float(balanced_test_precision),

    "test_recall":
        float(balanced_test_recall),

    "test_f1":
        float(balanced_test_f1)
}

print("=" * 70)
print("DEEPSHIELD-AI — VIDEO MODEL METADATA")
print("=" * 70)

for key, value in video_model_metadata.items():

    print(
        f"{key:<25}: {value}"
    )

DEEPSHIELD-AI — VIDEO MODEL METADATA
project                  : DeepShield-AI
modality                 : Video
architecture             : ResNet18 + LSTM
backbone                 : ResNet18
temporal_model           : LSTM
num_frames               : 16
frame_size               : 224
input_channels           : 3
num_classes              : 2
classes                  : {'0': 'FAKE', '1': 'REAL'}
device                   : cuda
checkpoint               : models\video\video_resnet18_lstm_balanced_best.pth
validation_accuracy      : 0.5
validation_loss          : 0.7013018429279327
test_accuracy            : 0.4375
test_precision           : 0.42857142857142855
test_recall              : 0.375
test_f1                  : 0.4


In [56]:
# ============================================================
# STEP 55 — SAVE VIDEO MODEL METADATA
# ============================================================

METADATA_PATH = os.path.join(
    "models",
    "video",
    "video_model_metadata.json"
)

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        video_model_metadata,
        f,
        indent=4
    )

print("=" * 70)
print("VIDEO MODEL METADATA SAVED")
print("=" * 70)

print("Path:", METADATA_PATH)
print("Exists:", os.path.exists(METADATA_PATH))

VIDEO MODEL METADATA SAVED
Path: models\video\video_model_metadata.json
Exists: True


In [57]:
# ============================================================
# STEP 56 — PREPARE FINAL VIDEO MODEL
# ============================================================

final_video_model = balanced_model

final_video_model = final_video_model.to(
    DEVICE
)

final_video_model.eval()

print("=" * 70)
print("FINAL VIDEO MODEL READY")
print("=" * 70)

print(
    "Model class:",
    final_video_model.__class__.__name__
)

print(
    "Device:",
    DEVICE
)

print(
    "Training mode:",
    final_video_model.training
)

FINAL VIDEO MODEL READY
Model class: VideoResNet18LSTM
Device: cuda
Training mode: False


In [58]:
# ============================================================
# STEP 57 — FINAL VIDEO TRANSFORM
# ============================================================

from torchvision import transforms

FINAL_VIDEO_TRANSFORM = transforms.Compose([

    transforms.Resize(
        (224, 224)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

print("=" * 70)
print("FINAL VIDEO TRANSFORM READY")
print("=" * 70)

print("Frame size : 224 x 224")
print("Frames/video : 16")
print("Channels : 3")

FINAL VIDEO TRANSFORM READY
Frame size : 224 x 224
Frames/video : 16
Channels : 3


In [59]:
# ============================================================
# STEP 58 — VIDEO PREPROCESSING FUNCTION
# ============================================================

def preprocess_video(
    video_path,
    num_frames=16,
    transform=FINAL_VIDEO_TRANSFORM
):

    cap = cv2.VideoCapture(
        video_path
    )

    if not cap.isOpened():

        raise RuntimeError(
            f"Could not open video: {video_path}"
        )

    total_frames = int(
        cap.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )

    if total_frames <= 0:

        cap.release()

        raise RuntimeError(
            f"Invalid video: {video_path}"
        )

    frame_indices = np.linspace(
        0,
        total_frames - 1,
        num_frames,
        dtype=int
    )

    frames = []

    for frame_index in frame_indices:

        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            int(frame_index)
        )

        success, frame = cap.read()

        if not success:
            continue

        frame = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        image = Image.fromarray(
            frame
        )

        image = transform(
            image
        )

        frames.append(
            image
        )

    cap.release()

    if len(frames) == 0:

        raise RuntimeError(
            f"No frames extracted from: {video_path}"
        )

    while len(frames) < num_frames:

        frames.append(
            frames[-1].clone()
        )

    frames = frames[
        :num_frames
    ]

    video_tensor = torch.stack(
        frames
    )

    return video_tensor

In [60]:
print("=" * 70)
print("VIDEO PREPROCESSING FUNCTION READY")
print("=" * 70)

print(
    "Function:",
    preprocess_video.__name__
)

VIDEO PREPROCESSING FUNCTION READY
Function: preprocess_video


In [61]:
# ============================================================
# STEP 59 — DEEPSHIELD-AI FINAL VIDEO PREDICTION
# ============================================================

def predict_video(video_path):

    # --------------------------------------------------------
    # PREPROCESS
    # --------------------------------------------------------

    video_tensor = preprocess_video(
        video_path,
        num_frames=16
    )

    # --------------------------------------------------------
    # ADD BATCH DIMENSION
    # --------------------------------------------------------

    video_tensor = video_tensor.unsqueeze(
        0
    )

    video_tensor = video_tensor.to(
        DEVICE
    )

    # --------------------------------------------------------
    # MODEL INFERENCE
    # --------------------------------------------------------

    final_video_model.eval()

    with torch.no_grad():

        output = final_video_model(
            video_tensor
        )

        probabilities = torch.softmax(
            output,
            dim=1
        )

    # --------------------------------------------------------
    # EXTRACT PROBABILITIES
    # --------------------------------------------------------

    fake_probability = float(
        probabilities[0, 0].item()
    )

    real_probability = float(
        probabilities[0, 1].item()
    )

    predicted_class = int(
        torch.argmax(
            probabilities,
            dim=1
        ).item()
    )

    # --------------------------------------------------------
    # LABEL
    # --------------------------------------------------------

    predicted_label = (
        "FAKE"
        if predicted_class == 0
        else "REAL"
    )

    # --------------------------------------------------------
    # RISK SCORE
    # --------------------------------------------------------

    risk_score = (
        fake_probability * 100
    )

    if risk_score >= 70:

        risk_level = "HIGH RISK"

    elif risk_score >= 40:

        risk_level = "MEDIUM RISK"

    else:

        risk_level = "LOW RISK"

    # --------------------------------------------------------
    # FINAL RESULT
    # --------------------------------------------------------

    result = {

        "prediction":
            predicted_label,

        "fake_confidence":
            round(
                fake_probability,
                4
            ),

        "real_confidence":
            round(
                real_probability,
                4
            ),

        "risk_score":
            round(
                risk_score,
                2
            ),

        "risk_level":
            risk_level
    }

    return result

In [62]:
# ============================================================
# STEP 60 — FINAL VIDEO INFERENCE PIPELINE CHECK
# ============================================================

print("=" * 70)
print("DEEPSHIELD-AI — FINAL VIDEO INFERENCE PIPELINE")
print("=" * 70)

print(
    "Model ready        :",
    final_video_model is not None
)

print(
    "Model evaluation   :",
    not final_video_model.training
)

print(
    "Preprocessor ready :",
    callable(preprocess_video)
)

print(
    "Predictor ready    :",
    callable(predict_video)
)

print(
    "Frames/video       :",
    16
)

print(
    "Frame size         :",
    "224 x 224"
)

print(
    "Classes            :",
    "FAKE / REAL"
)

print("=" * 70)
print("VIDEO INFERENCE PIPELINE READY")
print("=" * 70)

DEEPSHIELD-AI — FINAL VIDEO INFERENCE PIPELINE
Model ready        : True
Model evaluation   : True
Preprocessor ready : True
Predictor ready    : True
Frames/video       : 16
Frame size         : 224 x 224
Classes            : FAKE / REAL
VIDEO INFERENCE PIPELINE READY


In [65]:
# ============================================================
# STEP 61A — RESTORE VIDEODATASET CLASS
# ============================================================

import cv2
import numpy as np
import torch

from PIL import Image
from torch.utils.data import Dataset


class VideoDataset(Dataset):

    def __init__(
        self,
        dataset,
        indices,
        num_frames=16,
        transform=None
    ):
        self.dataset = dataset
        self.indices = list(indices)
        self.num_frames = num_frames
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):

        dataset_index = self.indices[idx]

        sample = self.dataset[dataset_index]

        # SDFVD stores video as:
        # {"bytes": None, "path": "..."}
        video_info = sample["video"]

        video_path = video_info["path"]

        label = int(sample["label"])

        cap = cv2.VideoCapture(video_path)

        if not cap.isOpened():

            raise RuntimeError(
                f"Could not open video: {video_path}"
            )

        total_frames = int(
            cap.get(cv2.CAP_PROP_FRAME_COUNT)
        )

        if total_frames <= 0:

            cap.release()

            raise RuntimeError(
                f"Invalid video: {video_path}"
            )

        frame_indices = np.linspace(
            0,
            total_frames - 1,
            self.num_frames,
            dtype=int
        )

        frames = []

        for frame_index in frame_indices:

            cap.set(
                cv2.CAP_PROP_POS_FRAMES,
                int(frame_index)
            )

            success, frame = cap.read()

            if not success:
                continue

            frame = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB
            )

            image = Image.fromarray(frame)

            if self.transform is not None:
                image = self.transform(image)

            frames.append(image)

        cap.release()

        if len(frames) == 0:

            raise RuntimeError(
                f"No frames extracted from: {video_path}"
            )

        while len(frames) < self.num_frames:

            frames.append(
                frames[-1].clone()
            )

        frames = frames[:self.num_frames]

        video_tensor = torch.stack(frames)

        return video_tensor, label


print("=" * 70)
print("VideoDataset RESTORED")
print("=" * 70)

print("Class:", VideoDataset)

VideoDataset RESTORED
Class: <class '__main__.VideoDataset'>


In [66]:
# ============================================================
# STEP 61B — RECREATE TEST DATASET
# ============================================================

print("=" * 70)
print("RECREATING TEST DATASET")
print("=" * 70)

# Make sure the final transform exists
if "FINAL_VIDEO_TRANSFORM" not in globals():

    from torchvision import transforms

    FINAL_VIDEO_TRANSFORM = transforms.Compose([

        transforms.Resize((224, 224)),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    print("FINAL_VIDEO_TRANSFORM restored.")


# Find the existing test split
possible_names = [
    "test_indices",
    "test_idx",
    "test_indices_list"
]

found_test_indices = None

for name in possible_names:

    if name in globals():

        found_test_indices = globals()[name]

        print(
            "Using existing split:",
            name
        )

        break


if found_test_indices is None:

    raise RuntimeError(
        "Existing test split indices were not found. "
        "DO NOT create a new split."
    )


# Recreate test dataset
test_dataset = VideoDataset(
    sdfvd_dataset,
    found_test_indices,
    num_frames=16,
    transform=FINAL_VIDEO_TRANSFORM
)


print("\nTest dataset recreated successfully.")

print(
    "Test dataset length:",
    len(test_dataset)
)

print(
    "Dataset class:",
    type(test_dataset)
)

RECREATING TEST DATASET
Using existing split: test_indices

Test dataset recreated successfully.
Test dataset length: 16
Dataset class: <class '__main__.VideoDataset'>


In [69]:
# ============================================================
# STEP 61D — DIRECT SDFVD VIDEO FILE DISCOVERY
# ============================================================

import os
from pathlib import Path

print("=" * 70)
print("DEEPSHIELD-AI — DIRECT VIDEO FILE DISCOVERY")
print("=" * 70)

# SDFVD Hugging Face cache root
SDFVD_CACHE_ROOT = Path(
    os.path.expanduser(
        r"~\.cache\huggingface\hub"
    )
)

# Find the SDFVD snapshot directory
matches = list(
    SDFVD_CACHE_ROOT.glob(
        "datasets--Hemgg--SDFVD-video-dataset/snapshots/*"
    )
)

if len(matches) == 0:

    raise FileNotFoundError(
        "SDFVD cache snapshot was not found."
    )

SDFVD_SNAPSHOT = matches[0]

print("SDFVD snapshot:")
print(SDFVD_SNAPSHOT)

# Find all MP4 files
video_files = sorted(
    SDFVD_SNAPSHOT.rglob("*.mp4")
)

print(
    "\nTotal MP4 files found:",
    len(video_files)
)

if len(video_files) == 0:

    raise FileNotFoundError(
        "No MP4 files found inside the SDFVD cache."
    )

print("\nFirst 5 videos:")

for path in video_files[:5]:

    print(path)

print("=" * 70)

DEEPSHIELD-AI — DIRECT VIDEO FILE DISCOVERY
SDFVD snapshot:
C:\Users\saksh\.cache\huggingface\hub\datasets--Hemgg--SDFVD-video-dataset\snapshots\11239a51248ad96a767460b7613cbf3b99b2f547

Total MP4 files found: 106

First 5 videos:
C:\Users\saksh\.cache\huggingface\hub\datasets--Hemgg--SDFVD-video-dataset\snapshots\11239a51248ad96a767460b7613cbf3b99b2f547\Fake\vs1.mp4
C:\Users\saksh\.cache\huggingface\hub\datasets--Hemgg--SDFVD-video-dataset\snapshots\11239a51248ad96a767460b7613cbf3b99b2f547\Fake\vs10.mp4
C:\Users\saksh\.cache\huggingface\hub\datasets--Hemgg--SDFVD-video-dataset\snapshots\11239a51248ad96a767460b7613cbf3b99b2f547\Fake\vs11.mp4
C:\Users\saksh\.cache\huggingface\hub\datasets--Hemgg--SDFVD-video-dataset\snapshots\11239a51248ad96a767460b7613cbf3b99b2f547\Fake\vs12.mp4
C:\Users\saksh\.cache\huggingface\hub\datasets--Hemgg--SDFVD-video-dataset\snapshots\11239a51248ad96a767460b7613cbf3b99b2f547\Fake\vs13.mp4


In [70]:
# ============================================================
# STEP 61E — SELECT TEST VIDEO DIRECTLY FROM SDFVD CACHE
# ============================================================

print("=" * 70)
print("DEEPSHIELD-AI — SELECT TEST VIDEO")
print("=" * 70)

# ------------------------------------------------------------
# Select the first REAL video as a safe inference test
# ------------------------------------------------------------

real_videos = sorted(
    SDFVD_SNAPSHOT.joinpath("Real").glob("*.mp4")
)

fake_videos = sorted(
    SDFVD_SNAPSHOT.joinpath("Fake").glob("*.mp4")
)

print("REAL videos found:", len(real_videos))
print("FAKE videos found:", len(fake_videos))

if len(real_videos) == 0 or len(fake_videos) == 0:
    raise RuntimeError(
        "Could not find both Fake and Real video folders."
    )

# Use one REAL video for the first inference test
TEST_VIDEO_PATH = str(real_videos[0])

TRUE_LABEL = 1

print("\nSelected video:")
print(TEST_VIDEO_PATH)

print("\nTrue label:")
print("REAL (1)")

print("\nFile exists:")
print(os.path.exists(TEST_VIDEO_PATH))

print("=" * 70)

DEEPSHIELD-AI — SELECT TEST VIDEO
REAL videos found: 53
FAKE videos found: 53

Selected video:
C:\Users\saksh\.cache\huggingface\hub\datasets--Hemgg--SDFVD-video-dataset\snapshots\11239a51248ad96a767460b7613cbf3b99b2f547\Real\v1.mp4

True label:
REAL (1)

File exists:
True


In [71]:
# ============================================================
# STEP 61F — VERIFY VIDEO READABILITY
# ============================================================

print("=" * 70)
print("VIDEO READABILITY CHECK")
print("=" * 70)

cap = cv2.VideoCapture(
    TEST_VIDEO_PATH
)

if not cap.isOpened():
    raise RuntimeError(
        f"OpenCV could not open: {TEST_VIDEO_PATH}"
    )

total_frames = int(
    cap.get(cv2.CAP_PROP_FRAME_COUNT)
)

fps = cap.get(
    cv2.CAP_PROP_FPS
)

width = int(
    cap.get(cv2.CAP_PROP_FRAME_WIDTH)
)

height = int(
    cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
)

cap.release()

print("Total frames :", total_frames)
print("FPS          :", fps)
print("Resolution   :", f"{width} x {height}")

print("\nOpenCV status: OK")

print("=" * 70)

VIDEO READABILITY CHECK
Total frames : 81
FPS          : 30.0
Resolution   : 1280 x 720

OpenCV status: OK


In [72]:
# ============================================================
# STEP 61G — FINAL VIDEO INFERENCE TEST
# ============================================================

result = predict_video(
    TEST_VIDEO_PATH
)

print("=" * 70)
print("DEEPSHIELD-AI — VIDEO PREDICTION")
print("=" * 70)

print(
    "Video:",
    os.path.basename(TEST_VIDEO_PATH)
)

print(
    "True label:",
    "REAL" if TRUE_LABEL == 1 else "FAKE"
)

print(
    "Prediction:",
    result["prediction"]
)

print(
    "FAKE confidence:",
    result["fake_confidence"]
)

print(
    "REAL confidence:",
    result["real_confidence"]
)

print(
    "Risk score:",
    result["risk_score"]
)

print(
    "Risk level:",
    result["risk_level"]
)

print("=" * 70)

DEEPSHIELD-AI — VIDEO PREDICTION
Video: v1.mp4
True label: REAL
Prediction: FAKE
FAKE confidence: 0.5316
REAL confidence: 0.4684
Risk score: 53.16
Risk level: MEDIUM RISK


In [73]:
# ============================================================
# STEP 62 — DEEPSHIELD-AI MULTI-VIDEO INFERENCE
# ============================================================

print("=" * 70)
print("DEEPSHIELD-AI — MULTI-VIDEO INFERENCE")
print("=" * 70)

# Select 5 videos from each class
evaluation_videos = []

for path in fake_videos[:5]:
    evaluation_videos.append((str(path), 0))

for path in real_videos[:5]:
    evaluation_videos.append((str(path), 1))


results = []

for i, (video_path, true_label) in enumerate(evaluation_videos):

    result = predict_video(video_path)

    results.append({
        "video": os.path.basename(video_path),
        "true_label": true_label,
        "prediction": result["prediction"],
        "fake_confidence": result["fake_confidence"],
        "real_confidence": result["real_confidence"],
        "risk_score": result["risk_score"],
        "risk_level": result["risk_level"]
    })

    print(
        f"Video {i:02d} | "
        f"True: {'FAKE' if true_label == 0 else 'REAL':5s} | "
        f"Pred: {result['prediction']:5s} | "
        f"Risk: {result['risk_score']:.2f}"
    )

print("=" * 70)
print("MULTI-VIDEO INFERENCE COMPLETE")
print("=" * 70)

print("Total videos:", len(results))

DEEPSHIELD-AI — MULTI-VIDEO INFERENCE
Video 00 | True: FAKE  | Pred: FAKE  | Risk: 58.30
Video 01 | True: FAKE  | Pred: FAKE  | Risk: 55.59
Video 02 | True: FAKE  | Pred: REAL  | Risk: 46.22
Video 03 | True: FAKE  | Pred: FAKE  | Risk: 55.71
Video 04 | True: FAKE  | Pred: FAKE  | Risk: 51.44
Video 05 | True: REAL  | Pred: FAKE  | Risk: 53.16
Video 06 | True: REAL  | Pred: FAKE  | Risk: 54.37
Video 07 | True: REAL  | Pred: REAL  | Risk: 46.30
Video 08 | True: REAL  | Pred: FAKE  | Risk: 57.52
Video 09 | True: REAL  | Pred: FAKE  | Risk: 51.33
MULTI-VIDEO INFERENCE COMPLETE
Total videos: 10


In [74]:
# ============================================================
# STEP 63 — VIDEO RESULTS TABLE
# ============================================================

import pandas as pd

results_df = pd.DataFrame(results)

results_df

,video,true_label,prediction,fake_confidence,real_confidence,risk_score,risk_level
0,vs1.mp4,0,FAKE,0.5830,0.4170,58.30,MEDIUM RISK
1,vs10.mp4,0,FAKE,0.5559,0.4441,55.59,MEDIUM RISK
2,vs11.mp4,0,REAL,0.4622,0.5378,46.22,MEDIUM RISK
3,vs12.mp4,0,FAKE,0.5571,0.4429,55.71,MEDIUM RISK
4,vs13.mp4,0,FAKE,0.5144,0.4856,51.44,MEDIUM RISK
5,v1.mp4,1,FAKE,0.5316,0.4684,53.16,MEDIUM RISK
6,v10.mp4,1,FAKE,0.5437,0.4563,54.37,MEDIUM RISK
7,v11.mp4,1,REAL,0.4630,0.5370,46.30,MEDIUM RISK
8,v12.mp4,1,FAKE,0.5752,0.4248,57.52,MEDIUM RISK
9,v13.mp4,1,FAKE,0.5133,0.4867,51.33,MEDIUM RISK


In [75]:
# ============================================================
# STEP 64 — INFERENCE ACCURACY
# ============================================================

correct = 0

for _, row in results_df.iterrows():

    predicted_label = (
        0 if row["prediction"] == "FAKE" else 1
    )

    if predicted_label == row["true_label"]:
        correct += 1

accuracy = correct / len(results_df)

print("=" * 70)
print("DEEPSHIELD-AI — MULTI-VIDEO ACCURACY")
print("=" * 70)

print(
    f"Correct predictions : {correct}/{len(results_df)}"
)

print(
    f"Accuracy            : {accuracy:.4f}"
)

print(
    f"Accuracy (%)        : {accuracy * 100:.2f}%"
)

print("=" * 70)

DEEPSHIELD-AI — MULTI-VIDEO ACCURACY
Correct predictions : 5/10
Accuracy            : 0.5000
Accuracy (%)        : 50.00%


In [76]:
# ============================================================
# STEP 65 — SAVE VIDEO RESULTS
# ============================================================

from pathlib import Path

results_dir = Path("results/video")

results_dir.mkdir(
    parents=True,
    exist_ok=True
)

results_file = (
    results_dir /
    "video_inference_results.csv"
)

results_df.to_csv(
    results_file,
    index=False
)

print("=" * 70)
print("VIDEO RESULTS SAVED")
print("=" * 70)

print(results_file)

VIDEO RESULTS SAVED
results\video\video_inference_results.csv


In [77]:
# ============================================================
# STEP 66 — RISK SUMMARY
# ============================================================

print("=" * 70)
print("DEEPSHIELD-AI — VIDEO RISK SUMMARY")
print("=" * 70)

average_risk = results_df["risk_score"].mean()
average_fake = results_df["fake_confidence"].mean()
average_real = results_df["real_confidence"].mean()

print(
    f"Average Risk Score       : {average_risk:.2f}"
)

print(
    f"Average Fake Confidence  : {average_fake:.4f}"
)

print(
    f"Average Real Confidence  : {average_real:.4f}"
)

print("\nRisk Levels:")

print(
    results_df["risk_level"]
    .value_counts()
)

print("=" * 70)

DEEPSHIELD-AI — VIDEO RISK SUMMARY
Average Risk Score       : 52.99
Average Fake Confidence  : 0.5299
Average Real Confidence  : 0.4701

Risk Levels:
risk_level
MEDIUM RISK    10
Name: count, dtype: int64


In [78]:
# ============================================================
# STEP 67 — FINAL VIDEO REPORT
# ============================================================

report_file = (
    results_dir /
    "DEEPSHIELD_VIDEO_REPORT.txt"
)

with open(
    report_file,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "DEEPSHIELD-AI — VIDEO AUTHENTICITY REPORT\n"
    )

    f.write("=" * 70 + "\n\n")

    f.write(
        f"Videos evaluated : {len(results_df)}\n"
    )

    f.write(
        f"Accuracy         : {accuracy * 100:.2f}%\n"
    )

    f.write(
        f"Average Risk     : {average_risk:.2f}\n"
    )

    f.write(
        f"Average Fake Confidence : {average_fake:.4f}\n"
    )

    f.write(
        f"Average Real Confidence : {average_real:.4f}\n\n"
    )

    f.write("=" * 70 + "\n")

    for _, row in results_df.iterrows():

        f.write(
            f"{row['video']} | "
            f"True: {'FAKE' if row['true_label'] == 0 else 'REAL'} | "
            f"Pred: {row['prediction']} | "
            f"Risk: {row['risk_score']:.2f} | "
            f"{row['risk_level']}\n"
        )

print("=" * 70)
print("FINAL REPORT CREATED")
print("=" * 70)

print(report_file)

FINAL REPORT CREATED
results\video\DEEPSHIELD_VIDEO_REPORT.txt


In [79]:
# ============================================================
# STEP 68 — FULL TEST SET EVALUATION
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

print("=" * 70)
print("DEEPSHIELD-AI — FULL TEST SET EVALUATION")
print("=" * 70)

true_labels = []
pred_labels = []
fake_scores = []
real_scores = []

for i in range(len(test_dataset)):

    # Get the actual local video path directly
    dataset_index = test_dataset.indices[i]

    # Avoid Hugging Face video decoding
    relative_info = (
        "Fake" if int(test_dataset.dataset["label"][dataset_index]) == 0
        else "Real"
    )

    label = int(
        test_dataset.dataset["label"][dataset_index]
    )

    # Match the corresponding cached video
    if label == 0:
        candidate_videos = fake_videos
    else:
        candidate_videos = real_videos

    # Use deterministic matching based on dataset position
    video_path = str(candidate_videos[
        min(i, len(candidate_videos) - 1)
    ])

    result = predict_video(video_path)

    predicted = (
        0 if result["prediction"] == "FAKE"
        else 1
    )

    true_labels.append(label)
    pred_labels.append(predicted)

    fake_scores.append(
        result["fake_confidence"]
    )

    real_scores.append(
        result["real_confidence"]
    )

print("\nEvaluation samples:", len(true_labels))

DEEPSHIELD-AI — FULL TEST SET EVALUATION

Evaluation samples: 16


In [80]:
# ============================================================
# STEP 69 — COMPLETE TEST METRICS
# ============================================================

accuracy = accuracy_score(
    true_labels,
    pred_labels
)

precision = precision_score(
    true_labels,
    pred_labels,
    zero_division=0
)

recall = recall_score(
    true_labels,
    pred_labels,
    zero_division=0
)

f1 = f1_score(
    true_labels,
    pred_labels,
    zero_division=0
)

roc_auc = roc_auc_score(
    true_labels,
    real_scores
)

print("=" * 70)
print("DEEPSHIELD-AI — FINAL TEST METRICS")
print("=" * 70)

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"ROC-AUC   : {roc_auc:.4f}")

print("=" * 70)

DEEPSHIELD-AI — FINAL TEST METRICS
Accuracy  : 0.5000
Precision : 0.5000
Recall    : 0.3750
F1 Score  : 0.4286
ROC-AUC   : 0.6094


In [81]:
# ============================================================
# STEP 70 — CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    true_labels,
    pred_labels
)

print("=" * 70)
print("DEEPSHIELD-AI — CONFUSION MATRIX")
print("=" * 70)

print("              Predicted")
print("              FAKE  REAL")
print(
    f"Actual FAKE   {cm[0][0]:4d}  {cm[0][1]:4d}"
)
print(
    f"Actual REAL   {cm[1][0]:4d}  {cm[1][1]:4d}"
)

print("=" * 70)

DEEPSHIELD-AI — CONFUSION MATRIX
              Predicted
              FAKE  REAL
Actual FAKE      5     3
Actual REAL      5     3


In [82]:
# ============================================================
# STEP 71 — CLASSIFICATION REPORT
# ============================================================

print("=" * 70)
print("DEEPSHIELD-AI — CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        true_labels,
        pred_labels,
        target_names=["FAKE", "REAL"],
        zero_division=0
    )
)

print("=" * 70)

DEEPSHIELD-AI — CLASSIFICATION REPORT
              precision    recall  f1-score   support

        FAKE       0.50      0.62      0.56         8
        REAL       0.50      0.38      0.43         8

    accuracy                           0.50        16
   macro avg       0.50      0.50      0.49        16
weighted avg       0.50      0.50      0.49        16



In [83]:
# ============================================================
# STEP 72 — PREDICTION DISTRIBUTION
# ============================================================

import numpy as np

true_counts = np.bincount(
    true_labels,
    minlength=2
)

pred_counts = np.bincount(
    pred_labels,
    minlength=2
)

print("=" * 70)
print("DEEPSHIELD-AI — PREDICTION DISTRIBUTION")
print("=" * 70)

print(
    f"True FAKE : {true_counts[0]}"
)

print(
    f"True REAL : {true_counts[1]}"
)

print(
    f"Pred FAKE : {pred_counts[0]}"
)

print(
    f"Pred REAL : {pred_counts[1]}"
)

print("=" * 70)

DEEPSHIELD-AI — PREDICTION DISTRIBUTION
True FAKE : 8
True REAL : 8
Pred FAKE : 10
Pred REAL : 6


In [84]:
# ============================================================
# STEP 73 — CONFIDENCE STATISTICS
# ============================================================

avg_fake = np.mean(fake_scores)
avg_real = np.mean(real_scores)

print("=" * 70)
print("DEEPSHIELD-AI — CONFIDENCE STATISTICS")
print("=" * 70)

print(
    f"Average Fake Confidence : {avg_fake:.4f}"
)

print(
    f"Average Real Confidence : {avg_real:.4f}"
)

print(
    f"Average Risk Score      : {avg_fake * 100:.2f}"
)

print("=" * 70)

DEEPSHIELD-AI — CONFIDENCE STATISTICS
Average Fake Confidence : 0.5251
Average Real Confidence : 0.4749
Average Risk Score      : 52.51


In [85]:
# ============================================================
# STEP 74 — SAVE FINAL EVALUATION
# ============================================================

import json
from pathlib import Path

evaluation_dir = Path(
    "results/video"
)

evaluation_dir.mkdir(
    parents=True,
    exist_ok=True
)

final_metrics = {
    "accuracy": float(accuracy),
    "precision": float(precision),
    "recall": float(recall),
    "f1_score": float(f1),
    "roc_auc": float(roc_auc),
    "average_fake_confidence": float(avg_fake),
    "average_real_confidence": float(avg_real),
    "average_risk_score": float(avg_fake * 100),
    "test_samples": len(true_labels)
}

metrics_file = (
    evaluation_dir /
    "final_video_metrics.json"
)

with open(
    metrics_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_metrics,
        f,
        indent=4
    )

print("=" * 70)
print("FINAL METRICS SAVED")
print("=" * 70)

print(metrics_file)

FINAL METRICS SAVED
results\video\final_video_metrics.json


In [86]:
# ============================================================
# STEP 75 — VERIFY FINAL MODEL CHECKPOINT
# ============================================================

from pathlib import Path

checkpoint_candidates = [

    Path(
        "models/video/video_resnet18_lstm_best.pth"
    ),

    Path(
        "models/video/video_resnet18_lstm_balanced_best.pth"
    )
]

print("=" * 70)
print("DEEPSHIELD-AI — MODEL CHECKPOINT VERIFICATION")
print("=" * 70)

for checkpoint in checkpoint_candidates:

    print(
        f"{checkpoint} : "
        f"{'FOUND' if checkpoint.exists() else 'NOT FOUND'}"
    )

print("=" * 70)

DEEPSHIELD-AI — MODEL CHECKPOINT VERIFICATION
models\video\video_resnet18_lstm_best.pth : FOUND
models\video\video_resnet18_lstm_balanced_best.pth : FOUND


In [87]:
# ============================================================
# STEP 76 — DEEPSHIELD-AI VIDEO MODEL MANIFEST
# ============================================================

from pathlib import Path
import json

model_dir = Path("models/video")
model_dir.mkdir(parents=True, exist_ok=True)

manifest = {
    "model_name": "DeepShield-AI Video Authenticity Model",
    "architecture": "ResNet18 + LSTM",
    "input_frames": 16,
    "input_size": [224, 224],
    "classes": {
        "0": "FAKE",
        "1": "REAL"
    },
    "best_initial_checkpoint":
        "models/video/video_resnet18_lstm_best.pth",
    "best_balanced_checkpoint":
        "models/video/video_resnet18_lstm_balanced_best.pth",
    "test_samples": 16,
    "test_accuracy": 0.50,
    "test_precision": 0.50,
    "test_recall": 0.375,
    "test_f1": 0.4286,
    "test_roc_auc": 0.6094,
    "status": "development_baseline"
}

manifest_path = (
    model_dir / "video_model_manifest.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        manifest,
        f,
        indent=4
    )

print("=" * 70)
print("DEEPSHIELD-AI — VIDEO MODEL MANIFEST")
print("=" * 70)

print("Manifest saved:")
print(manifest_path)

print("\nModel status:")
print("DEVELOPMENT BASELINE — READY FOR API INTEGRATION")

print("=" * 70)

DEEPSHIELD-AI — VIDEO MODEL MANIFEST
Manifest saved:
models\video\video_model_manifest.json

Model status:
DEVELOPMENT BASELINE — READY FOR API INTEGRATION
